In [ ]:
import numpy as np
import pandas as pd

In [ ]:
movies_df = pd.read_csv("tmdb_5000_movies.csv")
credits_df = pd.read_csv("tmdb_5000_credits.csv")

In [ ]:
credits_df

In [ ]:
movies_df

In [ ]:
movies_df.shape

In [ ]:
movies_df = movies_df.merge(credits_df, on="title")

In [ ]:
movies_df.shape

In [ ]:
movies_df.head(5)

In [ ]:
movies_df.info()

In [ ]:
movies_df = movies_df[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]]

In [ ]:
movies_df.head()

In [ ]:
movies_df.info()

In [ ]:
movies_df.isnull().sum()

In [ ]:
movies_df.dropna(inplace=True)

In [ ]:
movies_df.duplicated().sum()

In [ ]:
movies_df.iloc[0].genres

In [ ]:
movies_df.iloc[0].keywords

In [ ]:
import ast

In [ ]:
def convert(text):
    try:
        return [
            item["name"]
            for item in ast.literal_eval(text)
            if "name" in item
        ]
    except:
        return []

In [ ]:
movies_df["genres"] = movies_df["genres"].apply(convert)
movies_df["keywords"] = movies_df["keywords"].apply(convert)

movies_df.head()

In [ ]:
def convert3(text):
    try:
        L = []
        counter = 0
        for i in ast.literal_eval(text):
            if counter < 3:
                L.append(i['name'])
                counter += 1
            else:
                break
        return L 
    except (ValueError, TypeError):
        return []

In [ ]:
movies_df["cast"] = movies_df["cast"].apply(convert3)
movies_df.head()

In [ ]:
def fetch_director(text):
    try:
        for item in ast.literal_eval(text):
            if item.get("job") == "Director":
                return [item["name"]]
        return []
    except:
        return []

In [ ]:
movies_df["crew"] = movies_df["crew"].apply(fetch_director)

In [ ]:
movies_df

In [ ]:
movies_df["overview"][0]

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9 ]", "", text)
    return text.split()

movies_df["overview"] = movies_df["overview"].apply(clean_text)

In [ ]:
movies_df.head()

In [ ]:
movies_df["genres"] = movies_df["genres"].apply(lambda x:[i.replace(" ", "") for i in x])
movies_df["keywords"] = movies_df["keywords"].apply(lambda x:[i.replace(" ", "") for i in x])
movies_df["cast"] = movies_df["cast"].apply(lambda x:[i.replace(" ", "") for i in x])
movies_df["crew"] = movies_df["crew"].apply(lambda x:[i.replace(" ", "") for i in x])

In [ ]:
movies_df

In [ ]:
movies_df["tags"] = movies_df["overview"] + movies_df["genres"] + movies_df["keywords"] + movies_df["cast"] + movies_df["crew"]
movies_df.head()

In [ ]:
new_df = movies_df[["movie_id", "title", "tags"]].copy()
new_df.head()

In [ ]:
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x))
new_df.head()

In [ ]:
new_df["tags"][0]

In [ ]:
new_df["tags"] = new_df["tags"].apply(lambda x:x.lower())
new_df["tags"][0]

In [ ]:
import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)


In [ ]:
new_df["tags"] = new_df["tags"].apply(stem)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words="english")

In [ ]:
vectors = tfidf.fit_transform(new_df["tags"])

In [ ]:
vectors[0]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity(vectors).shape

In [ ]:
similarity = cosine_similarity(vectors).astype(np.float32)

In [ ]:
similarity[0]

In [ ]:
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x:x[1])[1:6]

In [ ]:
def recommend(movie):
    match = new_df[new_df["title"].str.lower() == movie.lower()]

    if match.empty:
        print(f"Error: '{movie}' was not found in the movie database.")
        print("Please check your spelling or try another title.")
        return
        
    movie_index = match.index[0]
    distances = similarity[movie_index]

    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    print(f"Because you watched '{match.iloc[0].title}', we recommend:")
    for i in movies_list:
        print(f"{new_df.iloc[i[0]].title}")

In [ ]:
recommend("interstellar")

In [ ]:
import pickle

In [ ]:
pickle.dump(new_df, open("movies.pkl", "wb"))

In [ ]:
pickle.dump(new_df.to_dict(), open("movies_dict.pkl", "wb"))

In [ ]:
pickle.dump(similarity, open("similarity.pkl", "wb"))